# Qwen3-32B Probe-Vector Logit Lens

This notebook implements a simple probe-vector logit lens for the saved Qwen3-32B steering vectors.

If the probe vector is `e`, the notebook computes:

- `e^T @ W_U`

where `W_U` is the Qwen3-32B unembedding matrix. It then shows the top positive and top negative vocabulary tokens for each selected stored probe vector.

Important implementation detail:

- it only downloads the tokenizer, the model weight index, and the single shard containing the unembedding weights
- it does **not** load the full model

The printed items are vocabulary tokens, so some entries may be subword pieces rather than whole words.


In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from IPython.display import display
from safetensors import safe_open
from transformers import AutoTokenizer


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate repo root from current working directory.')


ROOT = find_repo_root(Path.cwd())
RESULTS_ROOT = ROOT / 'results'
OUTPUT_DIR = RESULTS_ROOT / 'qwen3_32b' / 'probe_vector_logit_lens'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Repo root   :', ROOT)
print('Results root:', RESULTS_ROOT)
print('Output dir  :', OUTPUT_DIR)


/Users/michalmraz/opt/anaconda3/envs/temporal-awareness-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo root   : /Users/michalmraz/code/spar-ai/temporal-awareness
Results root: /Users/michalmraz/code/spar-ai/temporal-awareness/results
Output dir  : /Users/michalmraz/code/spar-ai/temporal-awareness/results/qwen3_32b/probe_vector_logit_lens


In [2]:
REQUIRED_ARTIFACT_FORMAT_VERSION = 1
MODEL_NAME = 'Qwen/Qwen3-32B'
TOP_K = int(globals().get('TOP_K', 10))
FILTER_ASCII_ENGLISH_ONLY = bool(globals().get('FILTER_ASCII_ENGLISH_ONLY', True))

ARTIFACT_PATH = globals().get('ARTIFACT_PATH', None)
METADATA_PATH = globals().get('METADATA_PATH', None)
SELECT_TRAIN_REGIMES = globals().get('SELECT_TRAIN_REGIMES', None)
SELECT_FEATURE_NAMES = globals().get('SELECT_FEATURE_NAMES', None)
SELECT_LAYERS = globals().get('SELECT_LAYERS', None)
SELECT_VECTOR_KEYS = globals().get('SELECT_VECTOR_KEYS', ['mm_probe_vectors', 'wmm_probe_vectors'])

SEARCH_ROOTS = [
    RESULTS_ROOT / 'qwen3_32b' / 'question_only_probe_variations',
    RESULTS_ROOT / 'qwen3_32b',
    RESULTS_ROOT,
]


def decode_str_array(values):
    decoded = []
    for value in values:
        if isinstance(value, bytes):
            decoded.append(value.decode('utf-8'))
        else:
            decoded.append(str(value))
    return decoded


def load_metadata(metadata_path):
    if metadata_path is None or not metadata_path.exists():
        return None
    return json.loads(metadata_path.read_text())


def metadata_is_compatible(metadata):
    if not metadata:
        return False
    return (
        int(metadata.get('artifact_format_version', 0)) >= REQUIRED_ARTIFACT_FORMAT_VERSION
        and metadata.get('model_name') == MODEL_NAME
        and metadata.get('prompt_family') == 'question_only_teacher_forced_answers'
        and metadata.get('explicit_split_granularity') == 'question'
        and metadata.get('implicit_split_granularity') == 'question'
        and bool(metadata.get('probe_prompt_use_chat_template', False))
        and bool(metadata.get('probe_prompt_disable_thinking_trace', metadata.get('disable_thinking_trace', False)))
    )


def resolve_metadata_path_for_artifact(artifact_path):
    candidate = artifact_path.with_name(artifact_path.name.replace('_probe_artifacts_', '_probe_metadata_').replace('.npz', '.json'))
    return candidate if candidate.exists() else None


def is_primary_artifact_path(artifact_path):
    return bool(re.fullmatch(r'qwen3_32b_question_only_probe_artifacts_\d{8}-\d{6}\.npz', artifact_path.name))


def find_latest_artifact(search_roots):
    artifact_candidates = []
    for root in search_roots:
        if root.exists():
            artifact_candidates.extend(sorted(root.rglob('qwen3_32b_question_only_probe_artifacts_*.npz')))

    primary_candidates = [path for path in artifact_candidates if is_primary_artifact_path(path)]
    checkpoint_candidates = [path for path in artifact_candidates if path not in primary_candidates]

    for artifact_path in sorted(primary_candidates, reverse=True) + sorted(checkpoint_candidates, reverse=True):
        metadata_path = resolve_metadata_path_for_artifact(artifact_path)
        metadata = load_metadata(metadata_path)
        if metadata_is_compatible(metadata):
            return artifact_path, metadata_path, metadata

    raise FileNotFoundError('Could not find a compatible Qwen3-32B probe artifact bundle under the configured search roots.')


def pick_default_one(requested, preferred, available):
    if requested is not None:
        return list(requested)
    if preferred in available:
        return [preferred]
    if not available:
        raise ValueError('No compatible values are available for selection.')
    return [available[0]]


if ARTIFACT_PATH is None:
    artifact_path, metadata_path, probe_metadata = find_latest_artifact(SEARCH_ROOTS)
else:
    artifact_path = Path(ARTIFACT_PATH).expanduser()
    metadata_path = Path(METADATA_PATH).expanduser() if METADATA_PATH is not None else resolve_metadata_path_for_artifact(artifact_path)
    probe_metadata = load_metadata(metadata_path)
    if not metadata_is_compatible(probe_metadata):
        raise ValueError(f'Artifact metadata is not compatible with the current Qwen3-32B pipeline: {metadata_path}')

print('Artifact path:', artifact_path)
print('Metadata path:', metadata_path)
print('Recommended regime     :', probe_metadata.get('recommended_time_utility_train_regime'))
print('Recommended feature    :', probe_metadata.get('recommended_time_utility_feature_name'))
print('Recommended vector key :', probe_metadata.get('recommended_time_utility_vector_key'))
print('Recommended layer      :', probe_metadata.get('recommended_time_utility_layer'))
print('ASCII-English filter   :', FILTER_ASCII_ENGLISH_ONLY)


Artifact path: /Users/michalmraz/code/spar-ai/temporal-awareness/results/qwen3_32b/question_only_probe_variations/20260410-093720/qwen3_32b_question_only_probe_artifacts_20260410-093720.npz
Metadata path: /Users/michalmraz/code/spar-ai/temporal-awareness/results/qwen3_32b/question_only_probe_variations/20260410-093720/qwen3_32b_question_only_probe_metadata_20260410-093720.json
Recommended regime     : None
Recommended feature    : None
Recommended vector key : None
Recommended layer      : None
ASCII-English filter   : True


In [3]:
probe_bundle = np.load(artifact_path, allow_pickle=False)

train_regimes = decode_str_array(probe_bundle['train_regimes'])
feature_names = decode_str_array(probe_bundle['feature_names'])
available_layers = [int(layer) for layer in probe_bundle['layers'].tolist()]
available_vector_keys = [key for key in ['mm_probe_vectors', 'wmm_probe_vectors'] if key in probe_bundle.files]

preferred_train_regime = probe_metadata.get('recommended_time_utility_train_regime') or 'explicit_train_only'
preferred_feature_name = probe_metadata.get('recommended_time_utility_feature_name') or 'mean_answer_tokens'
preferred_layer = probe_metadata.get('recommended_time_utility_layer')

resolved_train_regimes = pick_default_one(SELECT_TRAIN_REGIMES, preferred_train_regime, train_regimes)
resolved_feature_names = pick_default_one(SELECT_FEATURE_NAMES, preferred_feature_name, feature_names)
resolved_layers = list(SELECT_LAYERS) if SELECT_LAYERS is not None else ([int(preferred_layer)] if preferred_layer in available_layers else available_layers)
resolved_vector_keys = [key for key in SELECT_VECTOR_KEYS if key in available_vector_keys]

if not resolved_vector_keys:
    raise ValueError(f'None of the requested vector keys are available. Available keys: {available_vector_keys}')

print('Available train regimes:', train_regimes)
print('Available features     :', feature_names)
print('Available layers       :', available_layers)
print('Available vector keys  :', available_vector_keys)
print('Resolved train regimes :', resolved_train_regimes)
print('Resolved features      :', resolved_feature_names)
print('Resolved layers        :', resolved_layers)
print('Resolved vector keys   :', resolved_vector_keys)

vector_rows = []
selected_vectors = []
for train_regime in resolved_train_regimes:
    if train_regime not in train_regimes:
        raise ValueError(f'Train regime {train_regime!r} not found. Available: {train_regimes}')
    regime_idx = train_regimes.index(train_regime)

    for feature_name in resolved_feature_names:
        if feature_name not in feature_names:
            raise ValueError(f'Feature name {feature_name!r} not found. Available: {feature_names}')
        feature_idx = feature_names.index(feature_name)

        for layer in resolved_layers:
            if int(layer) not in available_layers:
                raise ValueError(f'Layer {layer!r} not found. Available: {available_layers}')
            layer_idx = available_layers.index(int(layer))

            for vector_key in resolved_vector_keys:
                vector = np.asarray(probe_bundle[vector_key][regime_idx, feature_idx, layer_idx, :], dtype=np.float32)
                vector_norm = float(np.linalg.norm(vector))
                vector_rows.append({
                    'train_regime': train_regime,
                    'feature_name': feature_name,
                    'vector_key': vector_key,
                    'layer': int(layer),
                    'vector_norm': vector_norm,
                })
                selected_vectors.append({
                    'train_regime': train_regime,
                    'feature_name': feature_name,
                    'vector_key': vector_key,
                    'layer': int(layer),
                    'vector': vector,
                    'vector_norm': vector_norm,
                })

selected_vectors_df = pd.DataFrame(vector_rows)
display(selected_vectors_df)
print('Selected vector count:', len(selected_vectors))


Available train regimes: ['explicit_train_only', 'implicit_train_only']
Available features     : ['mean_answer_tokens', 'last_answer_token']
Available layers       : [24, 28, 32, 36, 40, 44, 48]
Available vector keys  : ['mm_probe_vectors', 'wmm_probe_vectors']
Resolved train regimes : ['explicit_train_only']
Resolved features      : ['mean_answer_tokens']
Resolved layers        : [24, 28, 32, 36, 40, 44, 48]
Resolved vector keys   : ['mm_probe_vectors', 'wmm_probe_vectors']


,train_regime,feature_name,vector_key,layer,vector_norm
0,explicit_train_only,mean_answer_tokens,mm_probe_vectors,24,1.0
1,explicit_train_only,mean_answer_tokens,wmm_probe_vectors,24,1.0
2,explicit_train_only,mean_answer_tokens,mm_probe_vectors,28,1.0
3,explicit_train_only,mean_answer_tokens,wmm_probe_vectors,28,1.0
4,explicit_train_only,mean_answer_tokens,mm_probe_vectors,32,1.0
5,explicit_train_only,mean_answer_tokens,wmm_probe_vectors,32,1.0
6,explicit_train_only,mean_answer_tokens,mm_probe_vectors,36,1.0
7,explicit_train_only,mean_answer_tokens,wmm_probe_vectors,36,1.0
8,explicit_train_only,mean_answer_tokens,mm_probe_vectors,40,1.0
9,explicit_train_only,mean_answer_tokens,wmm_probe_vectors,40,1.0


Selected vector count: 14


In [4]:
def load_unembedding_tensor(model_name):
    try:
        index_path = hf_hub_download(repo_id=model_name, filename='model.safetensors.index.json')
        weight_map = json.loads(Path(index_path).read_text()).get('weight_map', {})
        for tensor_name in ['lm_head.weight', 'model.embed_tokens.weight']:
            if tensor_name in weight_map:
                shard_path = hf_hub_download(repo_id=model_name, filename=weight_map[tensor_name])
                with safe_open(shard_path, framework='pt') as handle:
                    return tensor_name, handle.get_tensor(tensor_name)
        raise KeyError('Neither lm_head.weight nor model.embed_tokens.weight found in model.safetensors.index.json.')
    except Exception:
        shard_path = hf_hub_download(repo_id=model_name, filename='model.safetensors')
        with safe_open(shard_path, framework='pt') as handle:
            for tensor_name in ['lm_head.weight', 'model.embed_tokens.weight']:
                if tensor_name in handle.keys():
                    return tensor_name, handle.get_tensor(tensor_name)
        raise KeyError('Neither lm_head.weight nor model.embed_tokens.weight found in model.safetensors.')


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
unembedding_name, unembedding_weight = load_unembedding_tensor(MODEL_NAME)
W_U = unembedding_weight.transpose(0, 1).contiguous()
TOKENIZER_VOCAB_SIZE = int(len(tokenizer))
MODEL_VOCAB_SIZE = int(W_U.shape[1])

print('Tokenizer vocab size :', TOKENIZER_VOCAB_SIZE)
print('Model vocab size     :', MODEL_VOCAB_SIZE)
print('Loaded tensor name   :', unembedding_name)
print('Raw tensor shape     :', tuple(unembedding_weight.shape))
print('W_U shape            :', tuple(W_U.shape))
print('W_U dtype            :', W_U.dtype)

ENGLISH_TOKEN_PATTERN = re.compile(r"^[A-Za-z][A-Za-z'-]*$")


def token_strings(token_id):
    token_id = int(token_id)
    if token_id < 0 or token_id >= TOKENIZER_VOCAB_SIZE:
        return f'<model_only_token_{token_id}>', ''
    token_text = tokenizer.convert_ids_to_tokens(token_id)
    decoded_text = tokenizer.decode([token_id], clean_up_tokenization_spaces=False)
    return token_text, decoded_text


def is_ascii_english_token(decoded_text):
    stripped = (decoded_text or '').strip()
    if not stripped:
        return False
    if not stripped.isascii():
        return False
    return bool(ENGLISH_TOKEN_PATTERN.fullmatch(stripped))


english_token_mask = torch.zeros(MODEL_VOCAB_SIZE, dtype=torch.bool)
for token_id in range(MODEL_VOCAB_SIZE):
    _, decoded_text = token_strings(token_id)
    english_token_mask[token_id] = is_ascii_english_token(decoded_text)

print('English-like ASCII token count:', int(english_token_mask.sum().item()))


Tokenizer vocab size : 151669
Model vocab size     : 151936
Loaded tensor name   : lm_head.weight
Raw tensor shape     : (151936, 5120)
W_U shape            : (5120, 151936)
W_U dtype            : torch.bfloat16
English-like ASCII token count: 70096


In [5]:
def select_unique_rows(candidate_pairs, direction, vector_row, top_k=10):
    rows = []
    seen = set()
    for token_id, score in candidate_pairs:
        _, decoded_text = token_strings(token_id)
        decoded_stripped = decoded_text.strip().lower()
        if not decoded_stripped:
            continue
        dedup_key = decoded_stripped
        if dedup_key in seen:
            continue
        seen.add(dedup_key)
        rows.append({
            'train_regime': vector_row['train_regime'],
            'feature_name': vector_row['feature_name'],
            'vector_key': vector_row['vector_key'],
            'layer': vector_row['layer'],
            'direction': direction,
            'rank': len(rows) + 1,
            'token_id': int(token_id),
            'decoded_stripped': decoded_stripped,
            'score': float(score),
        })
        if len(rows) >= top_k:
            break
    return rows


def top_and_bottom_token_rows(vector_row, top_k=10):
    vector = torch.tensor(vector_row['vector'], dtype=W_U.dtype)
    scores = torch.matmul(vector, W_U)
    candidate_k = min(max(top_k * 50, 500), scores.shape[0])

    if FILTER_ASCII_ENGLISH_ONLY:
        if not bool(english_token_mask.any()):
            raise ValueError('The ASCII-English token filter removed the entire vocabulary.')
        mask = english_token_mask.to(scores.device)
        positive_scores = scores.masked_fill(~mask, float('-inf'))
        negative_scores = scores.masked_fill(~mask, float('inf'))
        top_values, top_indices = torch.topk(positive_scores, k=candidate_k)
        bottom_values, bottom_indices = torch.topk(-negative_scores, k=candidate_k)
    else:
        top_values, top_indices = torch.topk(scores, k=candidate_k)
        bottom_values, bottom_indices = torch.topk(-scores, k=candidate_k)

    positive_pairs = list(zip(top_indices.tolist(), top_values.tolist()))
    negative_pairs = [
        (token_id, -float(value))
        for token_id, value in zip(bottom_indices.tolist(), bottom_values.tolist())
    ]

    rows = []
    rows.extend(select_unique_rows(positive_pairs, 'positive', vector_row, top_k=top_k))
    rows.extend(select_unique_rows(negative_pairs, 'negative', vector_row, top_k=top_k))
    return rows


def build_logit_lens_df(vector_rows, top_k=10):
    lens_rows = []
    for vector_row in vector_rows:
        lens_rows.extend(top_and_bottom_token_rows(vector_row, top_k=top_k))
    return pd.DataFrame(lens_rows)


def display_logit_lens_results(lens_df, vector_rows):
    for vector_row in vector_rows:
        header = (
            f'train_regime={vector_row["train_regime"]} | feature={vector_row["feature_name"]} | '
            f'vector_key={vector_row["vector_key"]} | layer={vector_row["layer"]} | norm={vector_row["vector_norm"]:.4f}'
        )
        print('\n' + '=' * len(header))
        print(header)
        print('=' * len(header))

        positive_df = lens_df.loc[
            (lens_df['train_regime'] == vector_row['train_regime'])
            & (lens_df['feature_name'] == vector_row['feature_name'])
            & (lens_df['vector_key'] == vector_row['vector_key'])
            & (lens_df['layer'] == vector_row['layer'])
            & (lens_df['direction'] == 'positive')
        ].sort_values('rank').reset_index(drop=True)
        negative_df = lens_df.loc[
            (lens_df['train_regime'] == vector_row['train_regime'])
            & (lens_df['feature_name'] == vector_row['feature_name'])
            & (lens_df['vector_key'] == vector_row['vector_key'])
            & (lens_df['layer'] == vector_row['layer'])
            & (lens_df['direction'] == 'negative')
        ].sort_values('rank').reset_index(drop=True)

        print('Top positive tokens:')
        display(positive_df[['rank', 'token_id', 'decoded_stripped', 'score']])
        print('Top negative tokens:')
        display(negative_df[['rank', 'token_id', 'decoded_stripped', 'score']])


def run_logit_lens_for_vector_key(vector_key):
    vector_rows = [row for row in selected_vectors if row['vector_key'] == vector_key]
    if not vector_rows:
        print(f'No selected vectors for {vector_key}.')
        return pd.DataFrame(), None

    lens_df = build_logit_lens_df(vector_rows, top_k=TOP_K)
    output_csv_path = OUTPUT_DIR / f'{artifact_path.stem}_{vector_key}_logit_lens.csv'
    lens_df.to_csv(output_csv_path, index=False)
    display_logit_lens_results(lens_df, vector_rows)
    print('Saved token-lens table:', output_csv_path)
    return lens_df, output_csv_path


In [6]:
mm_lens_df, mm_output_csv_path = run_logit_lens_for_vector_key('mm_probe_vectors')



train_regime=explicit_train_only | feature=mean_answer_tokens | vector_key=mm_probe_vectors | layer=24 | norm=1.0000
Top positive tokens:


,rank,token_id,decoded_stripped,score
0,1,4134,ago,0.084961
1,2,74397,enal,0.080566
2,3,61639,baugh,0.076660
3,4,71456,buff,0.075684
4,5,53537,edd,0.074219
5,6,19986,theless,0.074219
6,7,1595,pty,0.071777
7,8,77115,icho,0.071289
8,9,21323,etal,0.071289
9,10,9376,eg,0.070801


Top negative tokens:


,rank,token_id,decoded_stripped,score
0,1,63936,ogh,-0.085449
1,2,12765,aron,-0.082520
2,3,84569,oling,-0.082520
3,4,72399,laat,-0.082031
4,5,8199,ander,-0.078613
5,6,59466,eteor,-0.078125
6,7,70679,osti,-0.077637
7,8,85284,swing,-0.077148
8,9,40879,ariat,-0.076660
9,10,2412,ots,-0.076660



train_regime=explicit_train_only | feature=mean_answer_tokens | vector_key=mm_probe_vectors | layer=28 | norm=1.0000
Top positive tokens:


...


train_regime=explicit_train_only | feature=mean_answer_tokens | vector_key=mm_probe_vectors | layer=48 | norm=1.0000
Top positive tokens:


,rank,token_id,decoded_stripped,score
0,1,1293,long,0.091309
1,2,38544,toint,0.090820
2,3,88813,years,0.087891
3,4,74397,enal,0.081543
4,5,16779,toolstripmenuitem,0.081543
5,6,53095,inbackground,0.081543
6,7,1595,pty,0.080566
7,8,20125,andles,0.080078
8,9,18857,igne,0.075684
9,10,68008,ppelin,0.075195


Top negative tokens:


,rank,token_id,decoded_stripped,score
0,1,70679,osti,-0.087891
1,2,26383,glas,-0.084961
2,3,72724,jad,-0.081055
3,4,12765,aron,-0.078613
4,5,6534,situation,-0.074219
5,6,81453,yas,-0.073242
6,7,57080,orda,-0.072754
7,8,72399,laat,-0.072266
8,9,63115,paged,-0.069824
9,10,65362,izr,-0.068359


Saved token-lens table: /Users/michalmraz/code/spar-ai/temporal-awareness/results/qwen3_32b/probe_vector_logit_lens/qwen3_32b_question_only_probe_artifacts_20260410-093720_mm_probe_vectors_logit_lens.csv


In [7]:
wmm_lens_df, wmm_output_csv_path = run_logit_lens_for_vector_key('wmm_probe_vectors')



train_regime=explicit_train_only | feature=mean_answer_tokens | vector_key=wmm_probe_vectors | layer=24 | norm=1.0000
Top positive tokens:


,rank,token_id,decoded_stripped,score
0,1,56802,mor,0.097168
1,2,4040,deter,0.091309
2,3,1941,ook,0.090820
3,4,57541,abus,0.089844
4,5,64746,jes,0.089844
5,6,55319,jug,0.083496
6,7,95531,soap,0.082520
7,8,43364,slide,0.079102
8,9,10273,rub,0.078613
9,10,72187,nerd,0.076172


Top negative tokens:


,rank,token_id,decoded_stripped,score
0,1,19319,uras,-0.101562
1,2,1475,ased,-0.089844
2,3,74770,active,-0.086914
3,4,34538,bach,-0.085938
4,5,31799,inactive,-0.085938
5,6,28582,aho,-0.083496
6,7,56982,ibu,-0.083496
7,8,34456,uristic,-0.079590
8,9,93458,bose,-0.079102
9,10,37120,ulatory,-0.077148



train_regime=explicit_train_only | feature=mean_answer_tokens | vector_key=wmm_probe_vectors | layer=28 | norm=1.0000
Top positive tokens:


...


train_regime=explicit_train_only | feature=mean_answer_tokens | vector_key=wmm_probe_vectors | layer=48 | norm=1.0000
Top positive tokens:


,rank,token_id,decoded_stripped,score
0,1,47258,quine,0.095215
1,2,38081,reuse,0.088379
2,3,87208,slim,0.083496
3,4,1470,ql,0.083496
4,5,82821,azen,0.083008
5,6,42172,ere,0.081543
6,7,44126,ofsize,0.081055
7,8,56770,cascade,0.081055
8,9,95226,obsolete,0.081055
9,10,17820,pus,0.081055


Top negative tokens:


,rank,token_id,decoded_stripped,score
0,1,18093,closest,-0.081543
1,2,64748,acr,-0.079590
2,3,9788,unknown,-0.079590
3,4,53701,zos,-0.077148
4,5,35921,accur,-0.074707
5,6,3583,dev,-0.073730
6,7,40576,proc,-0.072754
7,8,19659,aque,-0.072266
8,9,68607,hovering,-0.071777
9,10,69247,elor,-0.070801


Saved token-lens table: /Users/michalmraz/code/spar-ai/temporal-awareness/results/qwen3_32b/probe_vector_logit_lens/qwen3_32b_question_only_probe_artifacts_20260410-093720_wmm_probe_vectors_logit_lens.csv
